<a href="https://colab.research.google.com/github/Arfa-Tariq/AstroPlanner-AI/blob/main/notebooks/03_celestial_visibility.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Celestial Visibility Engine (v2) — AstroPlanner AI

This notebook provides a powerful AI-driven tool designed to help astronomers (both amateur and professional) plan their observations. It computes and presents which deep-sky objects (from the NGC/IC catalog) and solar system bodies are observable from a given location over an upcoming 7-night window.

##  Features

-   **Personalized Visibility Analysis**: Generates a weekly observation plan tailored to your specific location (latitude, longitude) and telescope's aperture.
-   **Deep-Sky Object Catalog Integration**: Utilizes the OpenNGC catalog, filtering objects based on type, magnitude (brightness), and declination to match your equipment and location.
-   **Precise Solar System Body Tracking**: Uses Skyfield for accurate rise, set, and transit times for the Moon and planets over full 24-hour periods.
-   **Intelligent Night Window Detection**: Identifies true astronomical darkness for each night, ensuring optimal observing conditions.
-   **Robust Filtering & Scoring**: Applies filters for altitude constraints (e.g., objects must be 30° above the horizon), considers moon proximity warnings, and scores objects based on brightness and peak altitude.
-   **Diversity in Recommendations**: Implements type-based caps to ensure a variety of deep-sky objects in the recommendations, preventing the list from being dominated by one type.
-   **Detailed Output**: Provides a JSON output detailing visible objects for each night, including peak altitude, rise/set times, and observation period (night/day/both).
-   **Spot Check Summary**: Highlights the 'best' night with the most visible objects, breaking down solar system bodies and top deep-sky objects.

##  How It Works

The engine leverages advanced astronomical libraries and a carefully designed pipeline:

### Architecture
-   **Skyfield**: Used for highly precise geometric calculations of rise, set, and transit times for planets and the Moon, ensuring full 24-hour coverage without truncation issues.
-   **Astroplan**: Enables fast, vectorized observability checks for thousands of deep-sky object candidates simultaneously, greatly optimizing performance.
-   **Astropy**: Provides foundational tools for handling celestial coordinates, transformations, and time systems.

### Pipeline
1.  **Initial Filtering (Pre-Sky)**: Applies cheap, early filters based on object `Type`, `Magnitude` (telescope's limiting magnitude), and `Declination` (objects that never reach a minimum altitude from your latitude).
2.  **Vectorized Observability Check**: `Astroplan` efficiently checks all remaining deep-sky candidates against altitude constraints within the defined night window.
3.  **Altitude and Window Analysis**: For deep-sky objects, it calculates peak altitude and the observable window, providing honest rise/set labels (e.g., 'up at dusk', 'up at dawn').
4.  **Planetary & Lunar Geometry**: `Skyfield` computes precise geometric rise/set/transit times for planets and the Moon over a full 24-hour cycle.
5.  **Composite Ranking & Diversity**: Objects are scored based on their brightness and peak altitude. A diversity cap ensures a varied list of recommended objects, preventing over-representation of any single type.

##  Getting Started

To use this tool, follow these steps:

### Setup
1.  **Install Libraries**: The notebook will automatically install necessary Python libraries (`astropy`, `astroplan`, `skyfield`, `requests`).
2.  **Mount Google Drive**: Your Google Drive will be mounted to `/content/drive` to store astronomical data (like ephemeris files and object catalogs) and observation plans persistently.
3.  **Clone Repository**: The `AstroPlanner-AI` GitHub repository will be cloned, providing custom helper functions and models required by the engine.
4.  **Data Directory**: A `DATA_DIR` is set up in your Google Drive (`/content/drive/MyDrive/AstroPlanner/data`) to cache downloaded astronomical data, ensuring faster subsequent runs.

### Configuration
-   **`current_request.json`**: You need to provide a `current_request.json` file in your `DATA_DIR` (e.g., `/content/drive/MyDrive/AstroPlanner/data/current_request.json`). This file defines your `UserProfile` and `WeeklyPlanRequest`, including your observer's latitude, longitude, and telescope aperture.

### Running the Notebook
1.  **Execute Cells**: Run all cells in the notebook sequentially. The

setup cells will handle library installations and data downloads.
2.  **Output**: The script will compute visibility for the next 7 nights and save the results to `weekly_visibility.json` in your `DATA_DIR`.
3.  **Spot Check**: A 'spot check' section at the end of the notebook provides a summary of the best observing night.

##  Dependencies

-   `astropy`
-   `astroplan`
-   `skyfield`
-   `requests`
-   `numpy`
-   `pandas`

##  Acknowledgements

This engine makes extensive use of and builds upon the excellent capabilities provided by:

-   **Skyfield**: For ephemeris data and precise calculations.
-   **Astroplan**: For efficient observation planning.
-   **Astropy**: For fundamental astronomical computations.
-   **OpenNGC**: For the deep-sky object catalog data.


### Architecture
- **Skyfield** — true geometric rise/set/transit for planets and Moon
  over a full 24-hour window, no grid-truncation issue
- **Astroplan** — vectorized observability check for NGC candidates,
  fast batch filtering across thousands of objects
- **Astropy** — coordinate transforms and time systems



## Setup

In [3]:
!pip install astropy astroplan skyfield requests timezonefinder tzdata -q

import sys, os, json, requests, warnings
import numpy as np
import pandas as pd
from datetime import datetime, timedelta, timezone, date as date_type
from collections import defaultdict
from google.colab import drive

drive.mount('/content/drive')

!git clone https://github.com/Arfa-Tariq/AstroPlanner-AI.git 2>/dev/null || git -C /content/AstroPlanner-AI pull

sys.path.append('/content/AstroPlanner-AI/src')

DATA_DIR = '/content/drive/MyDrive/AstroPlanner/data'
os.makedirs(DATA_DIR, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Already up to date.


##Imports

In [4]:
# Astropy / Astroplan — NGC catalog observability checks
from astropy.coordinates import EarthLocation, SkyCoord, AltAz
from astropy.time import Time
import astropy.units as u
from astropy.utils.iers import conf as iers_conf
from astroplan import Observer, FixedTarget
from astroplan import AltitudeConstraint, is_observable

# Skyfield — precise planetary rise/set/transit over full 24hr window
from skyfield.api import load, wgs84, N, E
from skyfield import almanac

from models import WeeklyPlanRequest, UserProfile

## Suppress warnings

In [5]:
import os

# Disable IERS auto-download — Colab blocks it, bundled table is
# accurate enough for observing planning
iers_conf.auto_download = False
iers_conf.auto_max_age  = None

warnings.filterwarnings('ignore', message='.*IERS.*')
warnings.filterwarnings('ignore', message='.*NonRotation.*')
warnings.filterwarnings('ignore', message='.*failed to download.*')
warnings.filterwarnings('ignore', message='.*Angular separation.*')
warnings.filterwarnings('ignore', message='.*unable to download.*')

# Load Skyfield ephemeris using a Loader pointed at DATA_DIR so the
# .bsp file is cached to Drive and reused across sessions.
# load.open() was wrong — it returns a raw file handle, not a parsed
# ephemeris. Loader(directory)(filename) is the correct pattern.
from skyfield.api import Loader

skyfield_loader = Loader(DATA_DIR)
ts  = skyfield_loader.timescale()

eph_path = os.path.join(DATA_DIR, 'de421.bsp')

try:
    eph = skyfield_loader('de421.bsp')  # attempts to load or download
except ValueError as e:
    if "file starts with b'<!DOCTYP'" in str(e) and os.path.exists(eph_path):
        print(f"Corrupted ephemeris file found at {eph_path}. Deleting and retrying download.")
        os.remove(eph_path)
        eph = skyfield_loader('de421.bsp') # retry download after deletion
    else:
        raise # Re-raise if it's a different ValueError

print(f"Ephemeris loaded: {eph}")
print(f"Timescale loaded: {ts}")

Ephemeris loaded: Segments from kernel file 'de421.bsp':
  JD 2414864.50 - JD 2471184.50  (1899-07-28 through 2053-10-08)
      0 -> 1    SOLAR SYSTEM BARYCENTER -> MERCURY BARYCENTER
      0 -> 2    SOLAR SYSTEM BARYCENTER -> VENUS BARYCENTER
      0 -> 3    SOLAR SYSTEM BARYCENTER -> EARTH BARYCENTER
      0 -> 4    SOLAR SYSTEM BARYCENTER -> MARS BARYCENTER
      0 -> 5    SOLAR SYSTEM BARYCENTER -> JUPITER BARYCENTER
      0 -> 6    SOLAR SYSTEM BARYCENTER -> SATURN BARYCENTER
      0 -> 7    SOLAR SYSTEM BARYCENTER -> URANUS BARYCENTER
      0 -> 8    SOLAR SYSTEM BARYCENTER -> NEPTUNE BARYCENTER
      0 -> 9    SOLAR SYSTEM BARYCENTER -> PLUTO BARYCENTER
      0 -> 10   SOLAR SYSTEM BARYCENTER -> SUN
      3 -> 301  EARTH BARYCENTER -> MOON
      3 -> 399  EARTH BARYCENTER -> EARTH
      1 -> 199  MERCURY BARYCENTER -> MERCURY
      2 -> 299  VENUS BARYCENTER -> VENUS
      4 -> 499  MARS BARYCENTER -> MARS
Timescale loaded: <skyfield.timelib.Timescale object at 0x7b45d632ea50>


## Load request

In [6]:
with open(f'{DATA_DIR}/current_request.json') as f:
    plan_request = WeeklyPlanRequest.model_validate_json(f.read())

print(f"User     : {plan_request.user.name}")
print(f"Location : {plan_request.user.latitude}°N, {plan_request.user.longitude}°E")
print(f"Aperture : {plan_request.user.telescope.aperture_mm}mm")

User     : Andrew
Location : 24.9126°N, 67.0682°E
Aperture : 70.0mm


## Determine observer's local timezone

Every rise/set/transit time computed below comes out of Skyfield/Astropy
in UTC. Previously that UTC value was formatted and saved as a plain
`HH:MM` string with no timezone marker — silently wrong for any observer
not in UTC+0, including fields literally named `rise_time_local` that
were never actually converted. Derived once here from lat/lon (handles
DST correctly, unlike a fixed offset) and threaded through every
function that produces a displayed time.

In [7]:
from timezonefinder import TimezoneFinder
from zoneinfo import ZoneInfo

def get_observer_timezone(latitude: float, longitude: float) -> ZoneInfo:
    """
    Looks up the IANA timezone for the observing site from its
    coordinates. Falls back to fixed UTC if no timezone is found (e.g.
    international waters) rather than crashing — better to show UTC
    explicitly than to guess wrong.
    """
    tf = TimezoneFinder()
    tz_name = tf.timezone_at(lat=latitude, lng=longitude)
    if tz_name is None:
        print("  Warning: could not determine timezone from coordinates — falling back to UTC.")
        return ZoneInfo("UTC")
    return ZoneInfo(tz_name)


local_tz = get_observer_timezone(plan_request.user.latitude, plan_request.user.longitude)
print(f"Observer timezone: {local_tz}")

Observer timezone: Asia/Karachi


## Load and prefilter NGC catalog

In [8]:
USEFUL_OBJECT_TYPES = {
    'GX', 'OC', 'GC', 'BN', 'EN', 'RN',
    'PN', 'SNR', 'SC', 'CL+N', 'G+C',
}

def load_ngc_catalog() -> pd.DataFrame:
    """
    Downloads and loads the OpenNGC catalog. Cached to Drive after
    first download so subsequent runs are instant.
    """
    cache_path = f'{DATA_DIR}/ngc_catalog.csv'
    if not os.path.exists(cache_path):
        print("Downloading OpenNGC catalog...")
        url = "https://raw.githubusercontent.com/mattiaverga/OpenNGC/master/database_files/NGC.csv"
        r = requests.get(url, timeout=30)
        r.raise_for_status()
        with open(cache_path, 'w') as f:
            f.write(r.text)
    df = pd.read_csv(cache_path, sep=';', low_memory=False)
    cols = ['Name', 'Type', 'RA', 'Dec', 'V-Mag', 'B-Mag', 'MajAx', 'Common names']
    df = df[[c for c in cols if c in df.columns]]
    df['magnitude'] = pd.to_numeric(df['V-Mag'], errors='coerce')
    df.loc[df['magnitude'].isna(), 'magnitude'] = pd.to_numeric(
        df.loc[df['magnitude'].isna(), 'B-Mag'], errors='coerce'
    )
    print(f"Catalog loaded: {len(df)} total objects")
    return df


def compute_limiting_magnitude(aperture_mm: float) -> float:
    """Standard visual limiting magnitude: 2.1 + 5 * log10(aperture_mm)."""
    return 2.1 + 5 * np.log10(aperture_mm)


def prefilter_catalog(df: pd.DataFrame, user: UserProfile) -> pd.DataFrame:
    """
    Three cheap filters before any sky computation:
    1. Type filter  — keep only meaningful deep-sky object types
    2. Magnitude    — drop objects too faint for this telescope
    3. Declination  — drop objects never reaching >30° at this latitude
    """
    def parse_ra(s):
        try:
            h, m, sec = str(s).split(':')
            return float(h)*15 + float(m)*0.25 + float(sec)*(15/3600)
        except Exception:
            return np.nan

    def parse_dec(s):
        try:
            parts = str(s).split(':')
            sign = -1 if str(s).startswith('-') else 1
            return sign*(abs(float(parts[0])) + float(parts[1])/60 + float(parts[2])/3600)
        except Exception:
            return np.nan

    df = df.copy()
    df['ra_deg']  = df['RA'].apply(parse_ra)
    df['dec_deg'] = df['Dec'].apply(parse_dec)
    df = df.dropna(subset=['ra_deg', 'dec_deg'])

    lim_mag = compute_limiting_magnitude(user.telescope.aperture_mm)
    lat     = user.latitude

    before = len(df)
    df = df[df['Type'].isin(USEFUL_OBJECT_TYPES)]
    print(f"After type filter      : {len(df):5d}  (removed {before-len(df)})")

    before = len(df)
    df = df[df['magnitude'].isna() | (df['magnitude'] <= lim_mag)]
    print(f"After magnitude filter : {len(df):5d}  (removed {before-len(df)})  [limit={lim_mag:.1f}]")

    before = len(df)
    df['max_altitude'] = 90 - abs(lat - df['dec_deg'])
    df = df[df['max_altitude'] >= 30]
    print(f"After declination filter:{len(df):5d}  (removed {before-len(df)})")

    return df.reset_index(drop=True)


ngc_df      = load_ngc_catalog()
filtered_df = prefilter_catalog(ngc_df, plan_request.user)
print(f"\nCandidates for sky computation: {len(filtered_df)}")

Catalog loaded: 13969 total objects
After type filter      :   141  (removed 13821)
After magnitude filter :    75  (removed 66)  [limit=11.3]
After declination filter:   51  (removed 24)

Candidates for sky computation: 51


## Build observer and targets once

In [9]:
def build_observer(user: UserProfile) -> Observer:
    """Builds Astroplan Observer. Created once, reused across all 7 nights."""
    return Observer(
        location=EarthLocation(
            lat=user.latitude  * u.deg,
            lon=user.longitude * u.deg,
            height=0 * u.m
        ),
        name="observer"
    )


def build_target_list(df: pd.DataFrame) -> tuple[list, list]:
    """
    Converts filtered catalog to FixedTarget objects.
    Built once, reused across all 7 nights.
    """
    targets, rows = [], []
    for _, row in df.iterrows():
        try:
            coord = SkyCoord(ra=row['ra_deg']*u.deg, dec=row['dec_deg']*u.deg)
            targets.append(FixedTarget(coord=coord, name=str(row['Name'])))
            rows.append(row)
        except Exception:
            continue
    print(f"Built {len(targets)} FixedTarget objects (reused across all 7 nights)")
    return targets, rows


observer        = build_observer(plan_request.user)
targets, valid_rows = build_target_list(filtered_df)

Built 51 FixedTarget objects (reused across all 7 nights)


## Night window helper

In [10]:
def get_night_window(observer: Observer, date_str: str):
    """
    Returns (night_start, night_end) for astronomical twilight.
    Returns (None, None) if no astronomical darkness exists this night
    — relevant for high latitudes (>~48°N) in summer.
    """
    try:
        midnight = Time(f"{date_str} 23:59:00")
        night_start = observer.twilight_evening_astronomical(midnight, which='nearest')
        night_end   = observer.twilight_morning_astronomical(midnight, which='nearest')
        if (night_end - night_start).to(u.hour).value <= 0:
            return None, None
        return night_start, night_end
    except Exception:
        return None, None

##  Skyfield planet rise/set/transit (full 24hr window):

In [11]:
PLANETS = [
    ('moon',                'Moon',    'Moon',   None),
    ('jupiter barycenter',  'Jupiter', 'Planet', -2.9),
    ('saturn barycenter',   'Saturn',  'Planet',  0.7),
    ('mars barycenter',     'Mars',    'Planet',  1.0),
    ('venus barycenter',    'Venus',   'Planet', -4.5),
    ('mercury barycenter',  'Mercury', 'Planet', -0.5),
]


def get_planet_visibility_skyfield(
    user: UserProfile,
    date_str: str,
    local_tz: ZoneInfo,
) -> list[dict]:
    """
    Computes true geometric rise/set/transit for solar system bodies using
    Skyfield's almanac over a FULL 24-hour window — not clipped to the
    night window. This eliminates the grid-truncation bug where objects
    still above the horizon at dawn were incorrectly labeled as 'setting'
    at the dawn boundary.

    All displayed times are converted from Skyfield's native UTC to the
    observer's local timezone (local_tz) before formatting — previously
    these were UTC values formatted without conversion, silently wrong
    for any non-UTC observer.

    Each planet/Moon is classified by observable_period:
    - 'night' = above 30° only during astronomical darkness
    - 'day'   = above 30° only during daylight hours
    - 'both'  = above 30° across both periods
    Planets visible only in daytime are included — Moon and planets are
    legitimate daytime targets unlike deep-sky objects which require darkness.
    """
    lat = user.latitude
    lon = user.longitude

    t0 = ts.utc(int(date_str[:4]), int(date_str[5:7]), int(date_str[8:10]),     12, 0, 0)
    t1 = ts.utc(int(date_str[:4]), int(date_str[5:7]), int(date_str[8:10]) + 1, 12, 0, 0)

    skyfield_location = wgs84.latlon(lat * N, lon * E)
    earth             = eph['earth']
    observer_sf       = earth + skyfield_location

    astropy_night_start, astropy_night_end = get_night_window(observer, date_str)

    results = []
    for body_key, display_name, obj_type, typical_mag in PLANETS:
        try:
            body = eph[body_key]

            # True geometric rise/set over full 24hr window
            f = almanac.risings_and_settings(
                eph, body, skyfield_location, horizon_degrees=-0.5
            )
            times, events = almanac.find_discrete(t0, t1, f)

            rise_time = None
            set_time  = None
            for t_ev, ev in zip(times, events):
                dt_str = t_ev.utc_datetime().astimezone(local_tz).strftime('%H:%M')
                if ev == 1 and rise_time is None:
                    rise_time = dt_str
                elif ev == 0 and set_time is None:
                    set_time = dt_str

            # True transit (peak altitude) over full 24hr window
            t_grid = ts.utc(
                int(date_str[:4]), int(date_str[5:7]), int(date_str[8:10]),
                np.linspace(12, 36, 145)  # every 10 minutes over 24hrs
            )
            astrometric      = observer_sf.at(t_grid).observe(body).apparent()
            alt, az, _       = astrometric.altaz()
            peak_idx         = int(np.argmax(alt.degrees))
            peak_alt         = float(alt.degrees[peak_idx])
            transit_time     = t_grid[peak_idx].utc_datetime().astimezone(local_tz).strftime('%H:%M')

            # --- Observable period classification (internal — stays in UTC/geometry, not displayed) ---

            if astropy_night_start is not None:
                night_t_grid = ts.utc(
                    int(date_str[:4]), int(date_str[5:7]), int(date_str[8:10]),
                    np.linspace(
                        astropy_night_start.to_datetime().hour + astropy_night_start.to_datetime().minute / 60,
                        astropy_night_end.to_datetime().hour   + astropy_night_end.to_datetime().minute   / 60 + 24,
                        20
                    )
                )
                night_astro      = observer_sf.at(night_t_grid).observe(body).apparent()
                night_alt, _, _  = night_astro.altaz()
                max_night_alt    = float(np.max(night_alt.degrees))
                visible_at_night = max_night_alt >= 30
            else:
                visible_at_night = False

            day_t_grid      = ts.utc(
                int(date_str[:4]), int(date_str[5:7]), int(date_str[8:10]),
                np.linspace(6, 18, 25)
            )
            day_astro       = observer_sf.at(day_t_grid).observe(body).apparent()
            day_alt, _, _   = day_astro.altaz()
            max_day_alt     = float(np.max(day_alt.degrees))
            visible_at_day  = max_day_alt >= 30

            if not visible_at_night and not visible_at_day:
                continue

            if visible_at_night and visible_at_day:
                observable_period = "both"
            elif visible_at_night:
                observable_period = "night"
            else:
                observable_period = "day"

            results.append({
                "name"               : display_name,
                "common_name"        : display_name,
                "type"               : obj_type,
                "magnitude"          : typical_mag,
                "size_arcmin"        : None,
                "ra_deg"             : None,
                "dec_deg"            : None,
                "peak_altitude_deg"  : round(peak_alt, 1),
                "transit_time"       : transit_time,
                "rise_time"          : rise_time or "circumpolar",
                "set_time"           : set_time  or "circumpolar",
                "moon_separation_deg": None,
                "moon_warning"       : False,
                "transits_after_dawn": False,
                "is_solar_system"    : True,
                "observable_period"  : observable_period,
            })

        except Exception:
            continue

    return results

## Nightly visibility computation

In [12]:
def compute_nightly_visibility(
    observer   : Observer,
    targets    : list,
    valid_rows : list,
    date_str   : str,
    user       : UserProfile,
    local_tz   : ZoneInfo,
) -> list[dict]:
    """
    Computes observable NGC/IC objects for one night.

    Rise/set labels are honest about the night window boundary:
    - 'up at dusk'  = already above 30° when astronomical darkness began
    - 'up at dawn'  = still above 30° when astronomical darkness ended
    - HH:MM         = actually crossed the 30° threshold during the night,
                      converted to the observer's local timezone (local_tz)

    Peak altitude and transit time reflect the maximum within the
    observable window. 'transits_after_dawn' flags objects still
    climbing at dawn so the chatbot can communicate this correctly.

    Moon separation computed per-object at its peak time (not midnight)
    to fix ~3° positional drift error for objects near the 30° threshold.
    Moon separation stored as a flag rather than used as a hard filter —
    recommendation engine deprioritizes rather than hard-rejects.

    All NGC objects tagged observable_period='night' since deep-sky
    objects are physically unobservable in daylight regardless of equipment.
    """
    try:
        night_start, night_end = get_night_window(observer, date_str)
        if night_start is None:
            return []

        duration_hours = (night_end - night_start).to(u.hour).value
        n_steps        = max(int(duration_hours * 2), 2)
        time_grid      = night_start + np.linspace(0, duration_hours, n_steps) * u.hour
        # Converted to local_tz here — previously .to_datetime() produced a
        # naive datetime that's actually UTC internally, formatted as if
        # it were already local. That was the "rise_time_local" bug.
        time_labels    = [
            t.to_datetime().replace(tzinfo=timezone.utc).astimezone(local_tz).strftime('%H:%M')
            for t in time_grid
        ]

        # Altitude-only constraint — moon separation handled per-object below
        constraints = [AltitudeConstraint(min=30 * u.deg)]

        observable_mask = is_observable(
            constraints, observer, targets,
            time_range=[night_start, night_end]
        )

        # Moon position at midnight as baseline
        from astropy.coordinates import get_body
        moon_midnight = get_body('moon', Time(f"{date_str} 23:59:00"), ephemeris='builtin')
        moon_icrs     = SkyCoord(
            ra=moon_midnight.ra.deg   * u.deg,
            dec=moon_midnight.dec.deg * u.deg,
            frame='icrs'
        )

        visible = []
        for target, row, is_obs in zip(targets, valid_rows, observable_mask):
            if not is_obs:
                continue
            try:
                coord = target.coord
                altaz = coord.transform_to(
                    AltAz(obstime=time_grid, location=observer.location)
                )
                alts = altaz.alt.deg

                peak_idx  = int(np.argmax(alts))
                peak_alt  = float(alts[peak_idx])
                peak_time = time_labels[peak_idx]

                # Honest rise/set labels
                above          = alts >= 30
                rising_indices = np.where(above)[0]
                if len(rising_indices) == 0:
                    continue

                rise_time = (
                    "up at dusk"
                    if rising_indices[0] == 0
                    else time_labels[rising_indices[0]]
                )
                set_time = (
                    "up at dawn"
                    if rising_indices[-1] == n_steps - 1
                    else time_labels[rising_indices[-1]]
                )

                transits_after_dawn = (peak_idx == n_steps - 1)

                # Moon separation at this object's peak time
                moon_sep  = float(moon_icrs.separation(coord).deg)
                moon_warn = moon_sep < 30

                mag    = row['magnitude']
                common = str(row.get('Common names', '') or '').split(';')[0].strip() or None

                visible.append({
                    "name"               : str(row['Name']),
                    "common_name"        : common,
                    "type"               : str(row.get('Type', 'Unknown')),
                    "magnitude"          : float(mag) if not pd.isna(mag) else None,
                    "size_arcmin"        : float(row['MajAx']) if 'MajAx' in row and not pd.isna(row.get('MajAx')) else None,
                    "ra_deg"             : round(float(row['ra_deg']), 4),
                    "dec_deg"            : round(float(row['dec_deg']), 4),
                    "peak_altitude_deg"  : round(peak_alt, 1),
                    "peak_time_local"    : peak_time,
                    "rise_time_local"    : rise_time,
                    "set_time_local"     : set_time,
                    "transits_after_dawn": transits_after_dawn,
                    "moon_separation_deg": round(moon_sep, 1),
                    "moon_warning"       : moon_warn,
                    "is_solar_system"    : False,
                    "observable_period"  : "night",
                })

            except Exception:
                continue

        # Composite score: 50% brightness + 50% altitude
        for obj in visible:
            b = max(0, (15 - (obj['magnitude'] or 15)) / 15)
            a = obj['peak_altitude_deg'] / 90
            obj['_score'] = 0.5 * a + 0.5 * b

        visible.sort(key=lambda x: x['_score'], reverse=True)

        # Per-type diversity cap
        TYPE_CAPS = {
            'GX'  : 30, 'OC': 15, 'GC': 15,
            'BN'  : 8,  'EN': 8,  'RN': 5,
            'PN'  : 10, 'SNR': 5, 'SC': 3,
            'CL+N': 5,  'G+C': 3,
        }
        counts  = defaultdict(int)
        diverse = []
        for obj in visible:
            t = obj['type']
            if counts[t] < TYPE_CAPS.get(t, 5):
                diverse.append(obj)
                counts[t] += 1
            if len(diverse) >= 100:
                break

        for obj in diverse:
            obj.pop('_score', None)

        # Planets prepended — always highest priority, include day-visible ones too
        planets = get_planet_visibility_skyfield(user, date_str, local_tz) or []
        return planets + diverse

    except Exception as e:
        print(f"\n  Warning: visibility computation failed for {date_str}: {e}")
        return []

## Weekly loop

In [13]:
def get_weekly_visibility(
    user       : UserProfile,
    observer   : Observer,
    targets    : list,
    valid_rows : list,
    start_date : date_type,
    local_tz   : ZoneInfo,
) -> list[dict]:
    """
    Computes visibility for all 7 nights. Observer and target list
    passed in — built once in Cell 16, never rebuilt.

    Anchored to start_date (plan_request.generated_at, fixed once in
    notebook 01) rather than calling today() independently — otherwise
    running this notebook on a different calendar day than notebook 02
    produces two 7-day windows that don't fully overlap, silently
    dropping nights where they disagree.

    All displayed times use local_tz. Gracefully handles per-night
    failures so one bad night doesn't crash the entire weekly computation.
    """
    weekly = []

    for offset in range(7):
        date_str = (start_date + timedelta(days=offset)).strftime('%Y-%m-%d')
        print(f"Computing {date_str}...", end=" ")

        try:
            nightly = compute_nightly_visibility(
                observer, targets, valid_rows, date_str, user, local_tz
            )
        except Exception as e:
            print(f"ERROR: {e}")
            nightly = []

        if nightly is None:
            nightly = []

        n_planets = sum(1 for o in nightly if o.get('is_solar_system'))
        n_dso     = sum(1 for o in nightly if not o.get('is_solar_system'))
        print(
            f"{len(nightly)} objects  "
            f"({n_planets} planets/moon + {n_dso} deep-sky)"
        )

        weekly.append({
            "date"                : date_str,
            "day_offset"          : offset,
            "timezone"            : str(local_tz),
            "visible_object_count": len(nightly),
            "objects"             : nightly,
        })

    return weekly

## Run and save

In [14]:
weekly_visibility = get_weekly_visibility(
    plan_request.user, observer, targets, valid_rows, plan_request.generated_at, local_tz
)

with open(f'{DATA_DIR}/weekly_visibility.json', 'w') as f:
    json.dump(weekly_visibility, f, indent=2, default=str)

print(f"\nSaved to {DATA_DIR}/weekly_visibility.json")
print("\n=== Visibility Summary ===")
for night in weekly_visibility:
    print(f"{night['date']}: {night['visible_object_count']} total objects")

Computing 2026-07-16... 21 objects  (6 planets/moon + 15 deep-sky)
Computing 2026-07-17... 21 objects  (6 planets/moon + 15 deep-sky)
Computing 2026-07-18... 21 objects  (6 planets/moon + 15 deep-sky)
Computing 2026-07-19... 21 objects  (6 planets/moon + 15 deep-sky)
Computing 2026-07-20... 21 objects  (6 planets/moon + 15 deep-sky)
Computing 2026-07-21... 21 objects  (6 planets/moon + 15 deep-sky)
Computing 2026-07-22... 21 objects  (6 planets/moon + 15 deep-sky)

Saved to /content/drive/MyDrive/AstroPlanner/data/weekly_visibility.json

=== Visibility Summary ===
2026-07-16: 21 total objects
2026-07-17: 21 total objects
2026-07-18: 21 total objects
2026-07-19: 21 total objects
2026-07-20: 21 total objects
2026-07-21: 21 total objects
2026-07-22: 21 total objects


 ## Spot check

In [15]:
best = max(weekly_visibility, key=lambda n: n['visible_object_count'])
print(f"Best night: {best['date']}  ({best['visible_object_count']} objects, times shown in {best['timezone']})\n")

print("=== Solar System Bodies (true geometric times) ===")
for obj in best['objects']:
    if obj.get('is_solar_system'):
        period = obj.get('observable_period', 'night')
        period_label = {
            'night': '🌙 night only',
            'day'  : '☀️  day only',
            'both' : '🌓 day + night',
        }.get(period, period)
        print(
            f"  {obj['name']:8}  "
            f"peak={obj['peak_altitude_deg']}° at {obj['transit_time']}  "
            f"rises={obj['rise_time']}  sets={obj['set_time']}  "
            f"[{period_label}]"
        )

print("\n=== Top 15 Deep-Sky Objects ===")
dso = [o for o in best['objects'] if not o.get('is_solar_system')]
for obj in dso[:15]:
    moon_flag = "  ⚠ near moon" if obj.get('moon_warning') else ""
    dawn_flag = "  (transits after dawn)" if obj.get('transits_after_dawn') else ""
    print(
        f"  {obj['name']:10} {(obj['common_name'] or ''):25} "
        f"type={obj['type']:5} mag={str(obj['magnitude']):5} "
        f"peak={obj['peak_altitude_deg']}° at {obj['peak_time_local']}  "
        f"rise={obj['rise_time_local']}  set={obj['set_time_local']}"
        f"{moon_flag}{dawn_flag}"
    )

print("\n=== Object Type Distribution ===")
dist = defaultdict(int)
for o in best['objects']:
    dist[o['type']] += 1
for t, c in sorted(dist.items(), key=lambda x: -x[1]):
    print(f"  {t:10}: {c}")

print("\n=== Observable Period Breakdown ===")
for period, label in [('night', '🌙 night'), ('day', '☀️  day'), ('both', '🌓 both')]:
    count = sum(1 for o in best['objects'] if o.get('observable_period') == period)
    if count:
        print(f"  {label}: {count} objects")

Best night: 2026-07-16  (21 objects, times shown in Asia/Karachi)

=== Solar System Bodies (true geometric times) ===
  Moon      peak=73.6° at 15:20  rises=08:50  sets=21:07  [🌓 day + night]
  Jupiter   peak=84.8° at 13:20  rises=06:35  sets=19:58  [🌓 day + night]
  Saturn    peak=68.6° at 05:50  rises=23:42  sets=11:57  [🌙 night only]
  Mars      peak=87.3° at 09:40  rises=02:51  sets=16:23  [🌓 day + night]
  Venus     peak=74.4° at 15:30  rises=09:12  sets=21:53  [🌓 day + night]
  Mercury   peak=82.4° at 12:10  rises=05:33  sets=18:47  [🌓 day + night]

=== Top 15 Deep-Sky Objects ===
  NGC6960    Veil Nebula,Filamentary Nebula,Western Veil type=SNR   mag=7.0   peak=84.2° at 01:42  rise=21:22  set=up at dawn
  NGC6853    Dumbbell Nebula           type=PN    mag=7.4   peak=85.8° at 00:37  rise=up at dusk  set=up at dawn
  NGC6995    Eastern Veil,Network Nebula type=SNR   mag=7.0   peak=83.3° at 01:42  rise=21:22  set=up at dawn
  NGC6992    Eastern Veil,Network Nebula type=SNR   mag=7